In [2]:
# !pip install Kaggle

# # Create the .kaggle directory if it doesn't exist
# !mkdir -p ~/.kaggle

# 	# Create the kaggle.json file with your credentials
# 	# Replace 'YOUR_KAGGLE_USERNAME' and 'YOUR_KAGGLE_API_KEY' with your actual credentials
# kaggle_json_content = '{"username":"Naveen K Shijo","key":"KGAT_df821c8121c193c5c89ff456ab6ddf42"}'
# with open('/root/.kaggle/kaggle.json', 'w') as f:
#   f.write(kaggle_json_content)

# # Set the correct permissions for the kaggle.json file
# # !chmod 600 ~/.kaggle/kaggle.json

# print("kaggle.json created and configured successfully!")

# !kaggle datasets download -d "jp797498e/twitter-entity-sentiment-analysis"
# !unzip twitter-entity-sentiment-analysis.zip

In [3]:
!pip install emoji
# !pip install vaderSentiment

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emoji
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack


from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

from sklearn.metrics import classification_report


   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 608.4/608.4 kB 6.1 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'vaderSentiment'

<h3>Topic based sentiment classification </h3>

In [ ]:
df = pd.read_csv("twitter_training.csv", names = ['id', 'topic', 'sentiment', 'text'])
df.head()

In [ ]:
df.isna().sum()

In [ ]:
(df.duplicated().sum())/len(df)  *100

In [ ]:
df.dtypes

<h4>Data Cleaning</h4>

In [ ]:
df = df.drop('id', axis = 1)
df = df.dropna()
df = df.drop_duplicates()
df.head(3)

In [ ]:
df['sentiment'].nunique()

# Label consistency check: Looking whether same text appears with different sentiment labels

In [ ]:
inconsistency = df.groupby(['text', 'topic'])['sentiment'].nunique().sort_values()
# it is a pandas series
inconsistency


In [ ]:
type(inconsistency)

In [ ]:
inconsistent_texts = inconsistency[inconsistency > 1]
print(len(inconsistent_texts))   # no. of groups with more than 1 sentiment for the corresponding text and topic combination

In [ ]:
inconsistent_texts
# inconsistent_texts.index
inconsistent_texts.reset_index()
inconsistent_texts.reset_index()[['text', 'topic']]

In [ ]:
problem_rows = df.merge(
    inconsistent_texts.reset_index()[['text', 'topic']],
    on = ['text' , 'topic'],
    how = 'inner'
)

# It selects only those rows from df where (text, topic) is inconsistent

problem_rows
# problem_rows.sort_values(by = 'text')

# Class balance

In [ ]:
df.columns

In [ ]:
df['sentiment'].value_counts(normalize = True)

Here there is no problematic class imbalance. Thus models will work fine

# Topic leakage

Topic leakage matters, because if there are topics where there is majority class, then the model may be biased towards predicting the dominating class whenever it sees that topic. Instead of learning the pattern behind text -> sentiment.

In [ ]:
corr = pd.crosstab(df["topic"], df["sentiment"], normalize="index")
# type(corr)
corr

In [ ]:
corr[(corr['Irrelevant'] >0.5) |	(corr['Negative'] > 0.5) |	(corr['Neutral'] > 0.5) |	(corr['Positive']>0.5)]

Here, I will be proceeding with topic leakage without handling it. Let's see, how it will end up.

# Text normalization decisions

Remove URLs, @mentions, hashtags

Handle elongated words

# Handling URLs

Why remove URLs --> URLs rarely carry sentiment signal




In [ ]:
df[df['text'].str.contains(r"http\S+", regex = True)]['sentiment'].value_counts(normalize = True)

In [ ]:
df['text'] = df['text'].str.replace(r"https\S+|www\S+", "", regex = True)
# This removes:
#   http://...
#   https://...
#   www....

# Handling mentions(@)

In [ ]:
df['text'] = df['text'].str.replace(r"@\W+", "", regex = True)
df['text'] = df['text'].str.replace(r"#", "", regex = True)

# converting emojis to words
df['text'] = df['text'].apply(lambda x: emoji.demojize(x))

df['text'] = df['text'].str.replace(r"\s+", " ", regex = True).str.strip()

# "hello     world"   → "hello world"
# "foo\t\tbar"        → "foo bar"
# "line1\nline2"      → "line1 line2"

# "  hello world  "  → "hello world"

In [ ]:
df = df[df['sentiment'] != 'Irrelevant']

In [ ]:
plt.hist(df['text'].apply(len), bins = 100)   # plotting the distribution of character length in each text
plt.show()

# Train test split

In [ ]:
df.columns

In [ ]:
X = df.drop(columns = ['sentiment'])
y = df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

In [ ]:
# X_train = X_train['topic'] + " " + X_train['text']
# X_test = X_test['topic'] + " " + X_test['text']
X_train.shape

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ("text_transformation", TfidfVectorizer(max_features = 5000, ngram_range= (1,2)), 'text'),
        ("topic_transformation", OneHotEncoder(handle_unknown='ignore'), ["topic"])
    ]
)

# Manually implementing without ColumnTransformer

In [ ]:
# TF-IDF on text
tfidf = TfidfVectorizer(
    max_features = 5000,
    ngram_range = (1,2)
)

# Learn vocabulary from training text
X_train_text = tfidf.fit_transform(X_train['text'])

X_test_text = tfidf.transform(X_test['text'])
# it converts the test texts into TF-IDF vectors using the same vocabulary and IDF weights learnt from training data

In [ ]:
# One-Hot encode topic

ohe = OneHotEncoder(handle_unknown = 'ignore')
X_train_topic = ohe.fit_transform(X_train[['topic']])
X_test_topic = ohe.transform(X_test[['topic']])

In [ ]:
# Combine both feature matrices: Both outputs are sparse matrices, we must horizontally stack them

X_train_final = hstack([X_train_text, X_train_topic])   # shape: rows * (5000 + x), where x -> no. of unique values in 'topic'
X_test_final = hstack([X_test_text, X_test_topic])

In [ ]:
X_test_final
# This is what ColumnTransformer was doing.

Logistic Regression (TF-IDF)

In [ ]:
log_model = Pipeline([
    ("Preprocessing the data", preprocessor),
    ("Logistic regression model", LogisticRegression(max_iter = 1000))
])

In [ ]:
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

# SVM Model

In [ ]:
svm_model = Pipeline([
    ("Cleaning and transforming data", preprocessor),
    ("SVM Model fitting", LinearSVC())
])
svm_model.fit(X_train, y_train)

In [ ]:
y_pred_svm = svm_model.predict(X_test)
print(classification_report(y_test, y_pred_svm))

# VADER

In [ ]:
# analyzer = SentimentIntensityAnalyzer()

# def vader_predict(text):
#   score = analyzer.polarity_scores(text)['compound']
#   if score >= 0.05:
#     return "Positive"
#   elif score <= -0.05:
#     return "Negative"
#   else:
#     return "Neutral"

In [ ]:
# y_pred_vader = X_test.apply(vader_predict)
# print("VADER report")
# print(classification_report(y_test, y_pred_vader))

# LSTM

LSTM cannot use TF-IDF

LSTM cannot use sparse matrices (hstack)

👉 LSTM needs dense padded sequences

In [ ]:
# concatenating topic into text
X_train_combined = X_train['topic'] + " " + X_train['text']
X_test_combined = X_test['topic'] + " " + X_test['text']


In [ ]:
tokenizer = Tokenizer(num_words = 10000)
# it creates a tokenizer that will only keep the 10000 most frequent words in the vocabulary

tokenizer.fit_on_texts(X_train_combined)
# learns the vocabulary from training data. It assigns unique integer index to each word ranked by frequency

X_train_seq = tokenizer.texts_to_sequences(X_train_combined)
# converts each text into a list of integers using the learned vocabulary  (Unknown words-not in the top 10k are simply dropped)
X_test_seq = tokenizer.texts_to_sequences(X_test_combined)

# Sequences have variable lengths. Neural networks need fixed length inputs, so
X_train_pad = pad_sequences(X_train_seq, maxlen = 50, padding = 'post')
X_test_pad = pad_sequences(X_test_seq, maxlen = 50, padding = 'post')

In [ ]:
len(X_train_pad[100])

In [ ]:
# Encode labels

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [ ]:
# Build LSTM

lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=10000, output_dim = 128))
lstm_model.add(LSTM(64))
lstm_model.add(Dense(len(le.classes_), activation = 'softmax'))

lstm_model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

In [ ]:
lstm_model.fit(
    X_train_pad,
    y_train_enc,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

In [ ]:
y_pred_lstm_prob = lstm_model.predict(X_test_pad)
# It gives the probabilities for each text and topic belonging to each 3 label(class)


In [ ]:
# To select the label with maximum probability for each data point,
y_pred_lstm = np.argmax(y_pred_lstm_prob, axis = 1)
print(classification_report(y_test_enc, y_pred_lstm))

In [ ]:
# to convert back into original labels
y_pred_labels = le.inverse_transform(y_pred_lstm)
y_test_labels = le.inverse_transform(y_test_enc)

In [ ]:
print(classification_report(y_test_labels, y_pred_labels))

In [ ]:
print(len(y_test_enc))
print(len(y_pred_lstm))

In [ ]:
y_test_enc

In [ ]:
y_pred_lstm

Checking prediction

In [ ]:
X_test

In [ ]:
# Logistic regression model

query = 'You son of God'
topic = 'Religion'

input_data = np.array([[topic, query]])
df3 = pd.DataFrame(input_data, columns = ['topic', 'text'])

log_model.predict(df3)

In [ ]:
# SVM model

query = 'You son of God'
topic = 'Religion'

input_data = np.array([[topic, query]])
df3 = pd.DataFrame(input_data, columns = ['topic', 'text'])

svm_model.predict(df3)

In [ ]:
# LSTM model

query = 'We need to kill all mosquitos'
topic = 'hatred'

input = topic + " " + query
input_seq = tokenizer.texts_to_sequences([input])
input_pad = pad_sequences(input_seq, maxlen = 50, padding = 'post')

le.inverse_transform(np.argmax(lstm_model.predict(input_pad), axis = 1))
